# Day 23: Model Performance Dashboard and Evaluation Report
## YBI Foundation — Python AI & Data Science Internship
### Project: AI-Based Student Performance Prediction System

---

### 1. Day 23 Objectives & Milestone Overview

During Days 14–18, our project developed, tuned, and validated machine learning models for early-warning student performance tier classification:
- **Day 14:** 5-Fold Stratified Cross-Validation on training data ($N=519$) comparing 6 candidate models.
- **Day 15:** Hyperparameter tuning using GridSearchCV on training CV folds; Decision Tree established as champion.
- **Day 16:** Final model evaluation on held-out test data ($N=130$) and in-depth error analysis.
- **Day 17:** Standalone production pipeline encapsulation (`artifacts/student_performance_pipeline.joblib`).
- **Day 18:** 11-suite automated validation and comprehensive user guide documentation.
- **Days 19–22:** Interactive Streamlit web application (`app.py`), monitoring telemetry (`src/monitoring.py`), and cloud deployment readiness.

#### Day 23 Objectives:
1. **Evaluation Artifact Discovery:** Audit and load the authentic historical evaluation results produced across Days 14–18 without synthesizing data.
2. **Comparative Model Analysis:** Systematically compare baseline and tuned model performance using 5-Fold Stratified Cross-Validation Macro F1.
3. **Held-Out Test Set Metrics & Class-Level Assessment:** Analyze precision, recall, and F1 across `Low`, `Medium`, and `High` performance tiers.
4. **Empirical Error Analysis:** Derive the true 3x3 confusion matrix and quantify error severity (adjacent vs. non-adjacent misclassifications).
5. **Interactive Dashboard Development:** Integrate a multi-page Streamlit evaluation dashboard at `pages/2_Model_Performance_Dashboard.py`.
6. **Exportable Evaluation Reporting:** Generate automated audit reports in both Markdown and CSV formats for educational decision-makers.
7. **Verification & Testing:** Validate data handling, graceful degradation on missing files, and end-to-end dashboard execution.

> **Data Integrity Safeguard Notice:** Final exam mark `G3` is strictly excluded from feature inputs to prevent target leakage. Target definitions are preserved as Low ($G3 < 10$), Medium ($10 \le G3 < 15$), and High ($G3 \ge 15$). Cross-validation metrics and held-out test metrics are kept strictly separate.

In [1]:
# Run from a cloned repository, locally or in Google Colab.
from pathlib import Path
import os, sys, subprocess
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Repository ready:", root.name)


Repository ready: IYB_INTERNSHIP


---
### 2. Environment Verification & Results-File Discovery

We begin by inspecting the local repository to locate authentic evaluation results generated in earlier stages.

In [2]:
import os
import sys
from pathlib import Path

# Resolve project root
PROJECT_ROOT = Path(".").resolve()
if (PROJECT_ROOT / "artifacts").exists():
    pass
elif (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.evaluation_dashboard import find_evaluation_files

print("=" * 70)
print("DAY 23: EVALUATION ARTIFACT AUDIT & DISCOVERY")
print("=" * 70)
found_files = find_evaluation_files(PROJECT_ROOT)
found_count = 0
for key, p in found_files.items():
    if p and p.exists():
        found_count += 1
        print(f"[FOUND] {key:<24} -> {p.name} ({p.stat().st_size:,} bytes)")
    else:
        print(f"[MISSING] {key:<22} -> Not Found")

print("-" * 70)
print(f"Audit Summary: {found_count}/{len(found_files)} evaluation artifacts verified successfully.")

DAY 23: EVALUATION ARTIFACT AUDIT & DISCOVERY
[FOUND] cv_results               -> day_14_cross_validation_results.csv (1,320 bytes)
[FOUND] baseline_vs_tuned        -> day_15_baseline_vs_tuned.csv (1,356 bytes)
[FOUND] tuning_results           -> day_15_tuning_results.csv (1,617 bytes)
[FOUND] day_15_test_metrics      -> day_15_final_test_metrics.csv (185 bytes)
[FOUND] final_model_metrics      -> day_16_final_model_metrics.csv (483 bytes)
[FOUND] classification_report    -> day_16_classification_report.csv (452 bytes)
[FOUND] error_analysis           -> day_16_error_analysis.csv (15,785 bytes)
[FOUND] model_selection_report   -> day_16_model_selection_report.json (1,908 bytes)
[FOUND] day_17_test_metrics      -> day_17_test_metrics.csv (251 bytes)
[FOUND] pipeline_metadata        -> student_performance_metadata.json (3,163 bytes)
----------------------------------------------------------------------
Audit Summary: 10/10 evaluation artifacts verified successfully.


---
### 3. Loading and Validating Evaluation Data

We use the modular `src/evaluation_dashboard.py` utilities to ingest the evaluation files with rigorous schema checks.

In [3]:
from src.evaluation_dashboard import (
    load_cv_results,
    load_baseline_vs_tuned,
    load_tuning_results,
    load_classification_report,
    load_error_analysis,
    load_model_selection_report
)

print("Loading evaluation data:")
cv_df, err = load_cv_results()
print(f"- CV Results: {'OK (' + str(len(cv_df)) + ' models evaluated)' if err is None else 'Error: ' + err}")

base_df, err = load_baseline_vs_tuned()
print(f"- Baseline vs Tuned: {'OK (' + str(len(base_df)) + ' models compared)' if err is None else 'Error: ' + err}")

clf_df, err = load_classification_report()
print(f"- Classification Report: {'OK (' + str(len(clf_df)) + ' metric rows)' if err is None else 'Error: ' + err}")

err_df, err = load_error_analysis()
print(f"- Error Analysis Data: {'OK (' + str(len(err_df)) + ' test instances loaded)' if err is None else 'Error: ' + err}")

sel_meta, err = load_model_selection_report()
print(f"- Model Selection Report: {'OK (Champion: ' + str(sel_meta.get('selected_model')) + ')' if err is None else 'Error: ' + err}")

Loading evaluation data:
- CV Results: OK (6 models evaluated)
- Baseline vs Tuned: OK (6 models compared)
- Classification Report: OK (6 metric rows)
- Error Analysis Data: OK (130 test instances loaded)
- Model Selection Report: OK (Champion: Decision Tree)


---
### 4. Comparing Baseline and Tuned Models

The primary evaluation criterion across Days 14 and 15 was **Macro F1** under 5-Fold Stratified Cross-Validation on the 519-student training split. Because students in the High and Low performance tiers represent minority subgroups, Macro F1 prevents majority-class overfitting.

In [4]:
import pandas as pd

print("=" * 70)
print("BASELINE VS TUNED 5-FOLD STRATIFIED CV MACRO F1 COMPARISON")
print("=" * 70)
print(f"{'Model':<20} {'Baseline CV F1':<16} {'Tuned CV F1':<16} {'Gain (Δ F1)':<12}")
print("-" * 70)
for _, row in base_df.iterrows():
    m = row['Model']
    b = f"{row['Baseline Mean CV Macro F1']:.4f} ± {row['Baseline Macro F1 Standard Deviation']:.4f}"
    t = f"{row['Tuned Mean CV Macro F1']:.4f} ± {row['Tuned Macro F1 Standard Deviation']:.4f}"
    g = f"{row['Change in Mean CV Macro F1']:+.4f}"
    print(f"{m:<20} {b:<16} {t:<16} {g:<12}")
print("-" * 70)
print(f"Champion Architecture: {base_df.iloc[0]['Model']} (Tuned CV Macro F1: {base_df.iloc[0]['Tuned Mean CV Macro F1']:.4f})")

BASELINE VS TUNED 5-FOLD STRATIFIED CV MACRO F1 COMPARISON
Model                Baseline CV F1   Tuned CV F1      Gain (Δ F1) 
----------------------------------------------------------------------
Decision Tree        0.4797 ± 0.0483  0.5237 ± 0.0389  +0.0440     
Logistic Regression  0.4308 ± 0.0546  0.4758 ± 0.0446  +0.0450     
SVM                  0.3142 ± 0.0384  0.4731 ± 0.0371  +0.1589     
KNN                  0.3751 ± 0.0661  0.4543 ± 0.0720  +0.0792     
Gaussian Naive Bayes 0.3796 ± 0.0354  0.4103 ± 0.0441  +0.0308     
Random Forest        0.4092 ± 0.0640  0.4092 ± 0.0640  +0.0000     
----------------------------------------------------------------------
Champion Architecture: Decision Tree (Tuned CV Macro F1: 0.5237)


Let us visualize the baseline versus tuned CV Macro F1 scores, including error bars for the fold-level standard deviations.

In [5]:
import matplotlib.pyplot as plt
import numpy as np

models = base_df['Model'].tolist()
x = np.arange(len(models))
width = 0.35

b_vals = base_df['Baseline Mean CV Macro F1'].fillna(0)
t_vals = base_df['Tuned Mean CV Macro F1'].fillna(0)
b_err = base_df['Baseline Macro F1 Standard Deviation'].fillna(0)
t_err = base_df['Tuned Macro F1 Standard Deviation'].fillna(0)

fig, ax = plt.subplots(figsize=(8, 4.5))
rects1 = ax.bar(x - width/2, b_vals, width, yerr=b_err, label='Baseline CV Macro F1', color='#93C5FD', capsize=3)
rects2 = ax.bar(x + width/2, t_vals, width, yerr=t_err, label='Tuned CV Macro F1', color='#1D4ED8', capsize=3)

ax.set_ylabel('Mean 5-Fold CV Macro F1')
ax.set_title('Cross-Validation Macro F1: Baseline vs Tuned Models (with ±1 Std Dev)')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_24132\2982132682.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### 5. Visualizing Classification Metrics on Unseen Test Data

On Day 16, the champion Decision Tree was evaluated once on the isolated 130-student held-out test split. Here we inspect class-level precision, recall, and F1-score alongside aggregate macro and weighted averages.

In [6]:
print("=" * 70)
print("HELD-OUT TEST SET CLASSIFICATION REPORT (N=130)")
print("=" * 70)
print(f"{'Class / Metric':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<12}")
print("-" * 70)
for _, row in clf_df.iterrows():
    c = row['Class/Average']
    p = f"{row['precision']:.4f}"
    r = f"{row['recall']:.4f}"
    f = f"{row['f1-score']:.4f}"
    s = f"{int(row['support']) if pd.notnull(row['support']) else ''}"
    if c == 'accuracy':
        print("-" * 70)
    print(f"{c:<20} {p:<12} {r:<12} {f:<12} {s:<12}")
print("=" * 70)

HELD-OUT TEST SET CLASSIFICATION REPORT (N=130)
Class / Metric       Precision    Recall       F1-Score     Support     
----------------------------------------------------------------------
Low                  0.3214       0.4500       0.3750       20          
Medium               0.6795       0.6310       0.6543       84          
High                 0.3333       0.3077       0.3200       26          
----------------------------------------------------------------------
accuracy             0.5385       0.5385       0.5385       0           
macro avg            0.4447       0.4629       0.4498       130         
weighted avg         0.5552       0.5385       0.5445       130         


---
### 6. Analyzing Errors and Misclassification Severity

Using the authentic prediction records from `day_16_error_analysis.csv`, we construct the empirical confusion matrix and quantify adjacent versus non-adjacent misclassifications.

In [7]:
from src.evaluation_dashboard import compute_confusion_matrix, compute_error_breakdown

cm = compute_confusion_matrix(err_df)
eb = compute_error_breakdown(err_df)

print("=" * 70)
print("EMPIRICAL CONFUSION MATRIX (Actual vs. Predicted, N=130)")
print("=" * 70)
print(cm)
print("-" * 70)
print(f"Total Test Instances:        {eb['total']}")
print(f"Correct Predictions:         {eb['correct']} ({100-eb['error_rate']*100:.1f}%)")
print(f"Total Misclassifications:    {eb['errors']} ({eb['error_rate']*100:.1f}%)")
print(f"Adjacent Errors (Low↔Med):   {eb['adjacent_count']} ({eb['adjacent_pct']:.1f}% of errors)")
print(f"Non-Adjacent Errors (Low↔Hi): {eb['non_adjacent_count']} ({eb['non_adjacent_pct']:.1f}% of errors)")
print("=" * 70)

EMPIRICAL CONFUSION MATRIX (Actual vs. Predicted, N=130)
predicted_category  Low  Medium  High
actual_category                      
Low                   9       9     2
Medium               17      53    14
High                  2      16     8
----------------------------------------------------------------------
Total Test Instances:        130
Correct Predictions:         70 (53.8%)
Total Misclassifications:    60 (46.2%)
Adjacent Errors (Low↔Med):   56 (93.3% of errors)
Non-Adjacent Errors (Low↔Hi): 4 (6.7% of errors)


We now plot the empirical confusion matrix heatmap alongside the misclassification severity pie chart.

In [8]:
import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax1, annot_kws={'size': 12, 'weight': 'bold'})
ax1.set_xlabel('Predicted Academic Tier', fontweight='bold')
ax1.set_ylabel('Actual Academic Tier', fontweight='bold')
ax1.set_title('Test Set Confusion Matrix (N=130)')

# Error Severity Pie
labels = ['Adjacent Errors (Low↔Med, Med↔High)', 'Non-Adjacent Errors (Low↔High)']
sizes = [eb['adjacent_count'], eb['non_adjacent_count']]
colors = ['#60A5FA', '#F87171']
ax2.pie(sizes, explode=(0, 0.1), labels=labels, autopct='%1.1f%%', startangle=140, colors=colors)
ax2.set_title('Misclassification Severity Distribution')

plt.tight_layout()
plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_24132\2050620818.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### 7. Integrating the Interactive Streamlit Dashboard

The interactive dashboard is implemented at `pages/2_Model_Performance_Dashboard.py`. Streamlit automatically discovers it as a multi-page component alongside `app.py`.

Key capabilities in `pages/2_Model_Performance_Dashboard.py`:
- **Overview KPIs:** Model name, CV Macro F1, CV Accuracy, Test Accuracy, Test F1, and Generalization Gap.
- **Model Comparison:** Interactive toggle between Macro F1, CV Accuracy, and Hyperparameter search spaces.
- **Class-Level Metrics:** Per-tier breakdown of precision, recall, and support with visual comparison.
- **Error Browser:** Interactive inspection of all 130 test instances with filters for severe misclassifications.
- **Report Downloads:** Live export buttons for both Markdown and CSV evaluation summaries.
- **Bidirectional Navigation:** Seamless switching between the prediction app and evaluation dashboard.

In [9]:
import py_compile
dash_file = PROJECT_ROOT / "pages" / "2_Model_Performance_Dashboard.py"
print(f"Dashboard script verified at: {dash_file}")
print(f"File Size: {dash_file.stat().st_size:,} bytes")
py_compile.compile(str(dash_file), doraise=True)
print("Syntax compilation: Clean (0 errors)")

Dashboard script verified at: C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\IYB_INTERNSHIP\pages\2_Model_Performance_Dashboard.py
File Size: 23,893 bytes
Syntax compilation: Clean (0 errors)


---
### 8. Exporting Evaluation Reports (Markdown & CSV)

We generate the standardized audit report strings and verify their structure.

In [10]:
from src.evaluation_dashboard import generate_markdown_report, generate_csv_summary

md_rep = generate_markdown_report(sel_meta, clf_df, eb, base_df)
csv_rep = generate_csv_summary(sel_meta, clf_df)

print("=" * 70)
print("SAMPLE EXTRACT FROM GENERATED MARKDOWN REPORT")
print("=" * 70)
print("\n".join(md_rep.splitlines()[:16]))

csv_rows = csv_rep.strip().splitlines()
print(f"\nCSV Summary Records Generated: {len(csv_rows) - 1} rows")

SAMPLE EXTRACT FROM GENERATED MARKDOWN REPORT
# Model Evaluation and Performance Verification Report
## AI-Based Student Performance Prediction System
**YBI Foundation Python AI & Data Science Internship — Day 23**

---

### 1. Executive Summary & Validation Scope

- **Selected Architecture:** Decision Tree
- **Model Selection Criterion:** Mean 5-Fold Stratified CV Macro F1
- **Cross-Validation Macro F1 (Training):** 0.5237 ± 0.0389
- **Held-Out Test Set Accuracy:** 53.85%
- **Held-Out Test Set Macro F1:** 0.4498
- **Generalization Gap (Test F1 - CV F1):** -0.0739
- **Evaluation Partition:** 80% Training (519 samples) / 20% Held-Out Test (130 samples)
- **Target Leakage Safeguard:** Active (`G3` strictly excluded from feature space; target categorized into Low/Medium/High)

CSV Summary Records Generated: 23 rows


---
### 9. Automated Testing, Limitations, and Development Log

We execute the test suite across both `tests/test_evaluation_dashboard.py` and existing readiness tests.

In [11]:
import unittest
suite = unittest.defaultTestLoader.discover(str(PROJECT_ROOT / "tests"), pattern="test_evaluation_dashboard.py")
runner = unittest.TextTestRunner(verbosity=1)
res = runner.run(suite)
print(f"Tests run: {res.testsRun}, Errors: {len(res.errors)}, Failures: {len(res.failures)}")

.

.

.

.

.

.

... (5 outputs left out)

2026-11-02 14:03:05.137 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


.

----------------------------------------------------------------------
Ran 8 tests in 1.620s

OK


Tests run: 8, Errors: 0, Failures: 0


#### Limitations & Ethical Governance
1. **Observational Data Only:** The features represent demographic and behavioral surveys from 649 Portuguese students. Statistical correlations (e.g., between failures or alcohol intake and grades) do not prove causality.
2. **Strict Leakage Prevention:** Final exam `G3` is omitted from feature inputs.
3. **Absence of Intermediate Grades:** In true early-warning scenarios, midterm grades `G1` and `G2` are not yet available at student intake.
4. **Supportive Early Warning Only:** The system is engineered to help educators allocate supplementary tutoring and pastoral support, never for academic sorting or punitive decisions.

#### Day 23 Development Log:
- `src/evaluation_dashboard.py`: Created modular loader, confusion matrix computation, and report generators.
- `pages/2_Model_Performance_Dashboard.py`: Built Streamlit multi-page dashboard with 6 comprehensive sections.
- `app.py`: Integrated bidirectional navigation between prediction app and evaluation dashboard.
- `tests/test_evaluation_dashboard.py`: Created 8 comprehensive automated test suites (all 8 passed).
- `notebooks/Day_23_Model_Performance_Dashboard.ipynb`: Validated Colab-compatible documentation notebook.